In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/test_queries.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/sample_submission.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/documents.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/dataset-metadata.json
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/train_queries.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/qrels_train.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/baseline_submission.csv


In [2]:
import pandas as pd
import numpy as np
df = pd.read_csv("/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/documents.csv", index_col ="document_id")
test = pd.read_csv("/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/test_queries.csv")
df.head()
test.head()

,query_id,query
0,1001,How do I cope with drought and erratic rainfal...
1,1002,How can I adapt my farming to drought and erra...
2,1003,How does drought and erratic rainfall affect m...
3,1004,What is the risk of drought and erratic rainfa...
4,1005,How do I cope with heat stress on my farm?


In [3]:
# My plan is to use hybird method which involves
# bm25 + dense retrival + reranker
def create_search_content(row):
    title = str(row.get("title", " ")).strip()
    text = str(row.get("text", " ")).strip()
    source = str(row.get("source", " ")).strip()
    crop = str(row.get("crop", " ")).strip()
    country = str(row.get("country", " ")).strip()
    source_url = str(row.get("source_url", " ")).strip()
    
    return f"Crop {crop} | Country {country} | Title {title} | Text {text} | Source {source}"
df_upd=pd.DataFrame()
df["search_text"] = df.apply(create_search_content, axis = 1)
docs_ids = df.index.to_list()
corpus_texts = df['search_text'].tolist()

In [4]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

# ----------------------------------------------------
# 1. Multi-GPU Device Setup & Model Initialization
# ----------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_gpus = torch.cuda.device_count()
print(f"Using device: {device} | Active GPUs: {num_gpus}")

model_name = "BAAI/bge-reranker-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)

# Move model to device and wrap in DataParallel if multiple GPUs exist
model.to(device)
if num_gpus > 1:
    print(f"Distributing across {num_gpus} GPUs via DataParallel.")
    model = nn.DataParallel(model)

# ----------------------------------------------------
# 2. Build Training Pairs (Positives + Hard Negatives)
# ----------------------------------------------------
train_queries_df = pd.read_csv('/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/train_queries.csv')
qrels_train_df = pd.read_csv('/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/qrels_train.csv')

tq_id_col = 'QueryId' if 'QueryId' in train_queries_df.columns else train_queries_df.columns[0]
tq_text_col = 'Query' if 'Query' in train_queries_df.columns else train_queries_df.columns[1]

qrels_qid_col = 'QueryId' if 'QueryId' in qrels_train_df.columns else qrels_train_df.columns[0]
qrels_did_col = 'DocumentId' if 'DocumentId' in qrels_train_df.columns else qrels_train_df.columns[1]

query_dict = dict(zip(train_queries_df[tq_id_col], train_queries_df[tq_text_col]))
doc_dict = dict(zip(docs_ids, corpus_texts))
positive_docs_per_query = qrels_train_df.groupby(qrels_qid_col)[qrels_did_col].apply(set).to_dict()

all_doc_ids_set = set(docs_ids)
train_texts_q = []
train_texts_d = []
train_labels = []

for q_id, pos_ids in positive_docs_per_query.items():
    if q_id not in query_dict:
        continue
    q_text = str(query_dict[q_id])
    
    # 1. True Positives (label = 1.0)
    for p_id in pos_ids:
        if p_id in doc_dict:
            train_texts_q.append(q_text)
            train_texts_d.append(doc_dict[p_id])
            train_labels.append(1.0)
            
    # 2. Sample 4 Negatives per query (label = 0.0)
    negative_candidates = list(all_doc_ids_set - pos_ids)
    sampled_neg_ids = random.sample(negative_candidates, min(4, len(negative_candidates)))
    for n_id in sampled_neg_ids:
        train_texts_q.append(q_text)
        train_texts_d.append(doc_dict[n_id])
        train_labels.append(0.0)

print(f"Total training pairs constructed: {len(train_labels):,}")

# ----------------------------------------------------
# 3. Dataset & Multi-GPU DataLoader
# ----------------------------------------------------
class PairDataset(Dataset):
    def __init__(self, queries, docs, labels, tokenizer, max_length=256):
        self.queries = queries
        self.docs = docs
        self.labels = torch.tensor(labels, dtype=torch.float32)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        inputs = self.tokenizer(
            self.queries[idx],
            self.docs[idx],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )
        return {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0),
            "label": self.labels[idx]
        }

train_dataset = PairDataset(train_texts_q, train_texts_d, train_labels, tokenizer)

# Scale batch size by number of GPUs (32 per GPU)
effective_batch_size = 32 * max(1, num_gpus)
train_loader = DataLoader(train_dataset, batch_size=effective_batch_size, shuffle=True)

# ----------------------------------------------------
# 4. Multi-GPU Training Loop
# ----------------------------------------------------
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
criterion = nn.BCEWithLogitsLoss()
epochs = 4

total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.1),
    num_training_steps=total_steps
)

model.train()
for epoch in range(epochs):
    running_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits.squeeze(-1)
        
        loss = criterion(logits, labels)
        loss.backward()
        
        optimizer.step()
        scheduler.step()
        running_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss / len(train_loader):.4f}")

# ----------------------------------------------------
# 5. Save the Fine-Tuned Model to Kaggle Disk
# ----------------------------------------------------
save_dir = "./fine_tuned_bge_reranker"
os.makedirs(save_dir, exist_ok=True)

# Unwrap DataParallel before saving to retain standard model format
model_to_save = model.module if isinstance(model, nn.DataParallel) else model

model_to_save.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"Fine-tuning complete! Model saved successfully to '{save_dir}'.")

Using device: cuda | Active GPUs: 2


config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Distributing across 2 GPUs via DataParallel.
Total training pairs constructed: 5,426
Epoch 1/4 - Loss: 0.2847
Epoch 2/4 - Loss: 0.0862
Epoch 3/4 - Loss: 0.0660
Epoch 4/4 - Loss: 0.0517


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuning complete! Model saved successfully to './fine_tuned_bge_reranker'.


In [5]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# ----------------------------------------------------
# 1. Device Setup & Load Saved Model from Disk
# ----------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_gpus = torch.cuda.device_count()
print(f"Running inference on: {device} | GPUs available: {num_gpus}")

save_dir = "./fine_tuned_bge_reranker"

tokenizer = AutoTokenizer.from_pretrained(save_dir)
model = AutoModelForSequenceClassification.from_pretrained(save_dir)
model.to(device)

# Wrap in DataParallel for 2x faster multi-GPU batch inference
if num_gpus > 1:
    model = nn.DataParallel(model)

model.eval()

# ----------------------------------------------------
# 2. Build Test Query-Document Pairs
# ----------------------------------------------------
# Load test queries if not already in memory
# test = pd.read_csv('/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/test_queries.csv')

query_id_col = 'QueryId' if 'QueryId' in test.columns else ('query_id' if 'query_id' in test.columns else test.columns[0])
query_text_col = 'Query' if 'Query' in test.columns else ('query' if 'query' in test.columns else test.columns[1])

all_test_queries = []
all_test_docs = []

for _, row in test.iterrows():
    q_text = str(row[query_text_col]).strip()
    for doc_text in corpus_texts:
        all_test_queries.append(q_text)
        all_test_docs.append(doc_text)

print(f"Total test pairs to score: {len(all_test_queries):,}")

# ----------------------------------------------------
# 3. Test Dataset & Batched Inference
# ----------------------------------------------------
class InferencePairDataset(Dataset):
    def __init__(self, queries, docs, tokenizer, max_length=256):
        self.queries = queries
        self.docs = docs
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.queries)

    def __getitem__(self, idx):
        inputs = self.tokenizer(
            self.queries[idx],
            self.docs[idx],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )
        return {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0)
        }

# Batch size scaled for multi-GPU
eval_batch_size = 256 * max(1, num_gpus)
test_loader = DataLoader(
    InferencePairDataset(all_test_queries, all_test_docs, tokenizer),
    batch_size=eval_batch_size,
    shuffle=False
)

all_scores = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits.squeeze(-1)
        
        all_scores.extend(logits.cpu().numpy().tolist())

all_scores = np.array(all_scores)

# ----------------------------------------------------
# 4. Extract Top 5 Documents & Format Submission
# ----------------------------------------------------
num_docs = len(corpus_texts)
submission_rows = []

for q_idx, (_, row) in enumerate(test.iterrows()):
    q_id = row[query_id_col]
    
    # Slice this query's block of scores across all corpus documents
    start_idx = q_idx * num_docs
    end_idx = start_idx + num_docs
    q_scores = all_scores[start_idx:end_idx]
    
    # Get top 5 highest scoring document positions
    top_5_pos = np.argsort(q_scores)[::-1][:5]
    
    for pos in top_5_pos:
        submission_rows.append({
            'QueryId': q_id,
            'DocumentId': docs_ids[pos]
        })

sub_df = pd.DataFrame(submission_rows)

# Validate format
assert list(sub_df.columns) == ['QueryId', 'DocumentId'], "Header mismatch!"
assert len(sub_df) == len(test) * 5, f"Expected {len(test) * 5} rows, got {len(sub_df)}"
assert (sub_df.groupby('QueryId')['DocumentId'].nunique() == 5).all(), "Duplicate documents found in top 5!"

sub_df.to_csv('submission_finetuned_reranker.csv', index=False)
print("Saved 'submission_finetuned_reranker.csv' successfully!")
print(sub_df.head(10))

Running inference on: cuda | GPUs available: 2


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Total test pairs to score: 139,000
Saved 'submission_finetuned_reranker.csv' successfully!
   QueryId  DocumentId
0     1001           5
1     1001           3
2     1001           4
3     1001           2
4     1001           1
5     1002           3
6     1002           5
7     1002           4
8     1002           2
9     1002           1
